In [3]:
# 이름 확인 후 같으면 아이디 확인 후 중복 제거

import pandas as pd

# 1. CSV 파일 불러오기
# 'data.csv' 부분에 실제 파일 경로 및 파일명을 입력해주세요.
df = pd.read_csv('dbpia_fast_results.csv')

# 2. 중복 제거 로직 적용
# subset=['Author Name', 'Author ID']: 이름과 아이디 두 컬럼을 동시에 봅니다.
# 두 컬럼의 값이 모두 같아야만 중복으로 판단하여 하나만 남깁니다.
# (이름은 같지만 아이디가 다르면 삭제되지 않고 유지됩니다.)
df_cleaned = df.drop_duplicates(subset=['Author Name', 'Author ID'], keep='first')

# 3. 결과 확인 (상위 5개만 출력)
print(f"원본 데이터 개수: {len(df)}개")
print(f"중복 제거 후 개수: {len(df_cleaned)}개")
print(df_cleaned.head())

# 4. 결과를 새로운 CSV 파일로 저장
# encoding='utf-8-sig'는 엑셀에서 한글이 깨지지 않게 해줍니다.
df_cleaned.to_csv('repetition_removal_authors.csv', index=False, encoding='utf-8-sig')

원본 데이터 개수: 163698개
중복 제거 후 개수: 93760개
                                          Source URL Author Name  Author ID
0  https://www.dbpia.co.kr/journal/articleDetail?...         송재엽  547050745
1  https://www.dbpia.co.kr/journal/articleDetail?...         유해성  288444857
2  https://www.dbpia.co.kr/journal/articleDetail?...         서영빈  541513697
3  https://www.dbpia.co.kr/journal/articleDetail?...         이인섭  672918873
4  https://www.dbpia.co.kr/journal/articleDetail?...         오주현  435693775


In [1]:
# 아이디만 확인 후 중복 제거

import pandas as pd

# 1. CSV 파일 불러오기
# 'data.csv'는 사용하시는 실제 파일명으로 변경해주세요.
df = pd.read_csv('dbpia_extract_authors_top_70.csv')

# 2. 아이디(Author ID) 기준 중복 제거
# subset=['Author ID']: 이제 이름은 보지 않고, 아이디가 같으면 무조건 중복으로 봅니다.
# keep='first': 중복된 아이디 중 가장 위에 있는 행만 남기고 나머지는 삭제합니다.
df_cleaned = df.drop_duplicates(subset=['Author ID'], keep='first')

# 3. 결과 확인
print(f"원본 데이터 개수: {len(df)}개")
print(f"중복 제거 후 개수: {len(df_cleaned)}개")

# 4. 파일 저장
df_cleaned.to_csv('repetition_removal_by_id.csv', index=False, encoding='utf-8-sig')

원본 데이터 개수: 105116개
중복 제거 후 개수: 61664개


In [6]:
# 아이디는 같은데 이름이 다른 경우 추출

import pandas as pd

# 1. 파일 불러오기
df = pd.read_csv('dbpia_fast_results.csv')

# 2. 아이디별로 등록된 '이름'의 종류 개수를 셉니다.
# nunique()는 고유한 값의 개수를 세는 함수입니다.
id_counts = df.groupby('Author ID')['Author Name'].nunique()

# 3. 이름이 2개 이상인 아이디(범인)만 찾아냅니다.
problem_ids = id_counts[id_counts > 1].index

# 4. 해당 아이디를 가진 데이터를 출력해서 눈으로 확인합니다.
result = df[df['Author ID'].isin(problem_ids)].sort_values('Author ID')

if len(result) > 0:
    print("아이디는 같은데 이름이 다른 경우가 발견되었습니다:")
    print(result[['Author Name', 'Author ID']])
    
    # 결과를 파일로 저장해서 보고 싶다면 아래 주석을 해제하세요
    result.to_csv('diff_names.csv', index=False, encoding='utf-8-sig')
else:
    print("아이디가 같으면 이름도 모두 같습니다. (중복 제거 개수가 같아야 정상입니다)")

아이디는 같은데 이름이 다른 경우가 발견되었습니다:
          Author Name  Author ID
37647             김동욱      73756
28560             김동욱      73756
122269  Dong-Wook Kim      73756
66321             김동욱      73756
59780   Dong-Wook Kim      73756
...               ...        ...
6368              김기문  999235305
106377            김기문  999235305
61407             허지훈  999483385
137070    Ji-Hoon Huh  999483385
6133              허지훈  999483385

[19301 rows x 2 columns]


In [2]:
# 중복 아이디 중 이름 표기(한/영)가 다를 경우 한글로 지정

import pandas as pd
import re

# 1. 파일 불러오기
df = pd.read_csv('dbpia_extract_authors_top_70.csv')

# 2. 한글 포함 여부를 판별하는 함수 정의
def has_korean(text):
    # 텍스트가 비어있으면 False
    if pd.isna(text):
        return False
    # 정규표현식으로 '가'부터 '힣'까지 한글이 하나라도 포함되어 있는지 확인
    return bool(re.search('[가-힣]', str(text)))

# 3. 'is_korean'이라는 임시 컬럼을 만들어서 한글이 있으면 True, 없으면 False를 표시
df['is_korean'] = df['Author Name'].apply(has_korean)

# 4. 정렬하기 (핵심 단계!)
# is_korean 컬럼을 기준으로 내림차순 정렬하면 True(한글)가 False(영어 등)보다 위로 올라옵니다.
df_sorted = df.sort_values(by='is_korean', ascending=False)

# 5. 아이디 기준 중복 제거
# 이제 한글 이름이 맨 위에 있으므로, keep='first'를 하면 한글 이름이 남게 됩니다.
df_cleaned = df_sorted.drop_duplicates(subset=['Author ID'], keep='first')

# 6. 임시로 만든 컬럼 삭제 및 저장
df_cleaned = df_cleaned.drop(columns=['is_korean'])
df_cleaned.to_csv('rep_rem_authors_in_korean.csv', index=False, encoding='utf-8-sig')

print("작업 완료!")
print(f"전체 {len(df)}개 중 {len(df_cleaned)}개가 남았습니다.")
print("우선순위에 따라 한글 이름으로 정리되었습니다.")

작업 완료!
전체 105116개 중 61664개가 남았습니다.
우선순위에 따라 한글 이름으로 정리되었습니다.


In [3]:
import pandas as pd
import json
import os

# 1. 파일 불러오기
csv_filename = 'dbpia_extract_authors_top_70.csv' 
json_filename = 'SSU_Datathon2025_공학분야_62199_Top70.json'

# 파일이 실제로 있는지 확인
if not os.path.exists(csv_filename):
    print(f"오류: '{csv_filename}' 파일을 찾을 수 없습니다. 현재 폴더를 확인해주세요.")
    exit()
if not os.path.exists(json_filename):
    print(f"오류: '{json_filename}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 데이터 로드
df = pd.read_csv(csv_filename)

with open(json_filename, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# 2. CSV에서 논문 ID 추출 및 매핑 테이블 생성
# URL 끝의 NODE_ID를 추출하여 연결고리로 사용
df['Paper_ID'] = df['Source URL'].apply(lambda x: x.split('nodeId=')[-1] if isinstance(x, str) and 'nodeId=' in x else None)

# 논문 ID별로 저자 ID들을 리스트로 묶음
paper_author_map = df.groupby('Paper_ID')['Author ID'].apply(list).to_dict()

# 3. JSON 데이터 업데이트 (메모리 상에서 처리)
# json_data가 리스트인지 딕셔너리인지 확인하여 유연하게 처리
if isinstance(json_data, dict):
    node_list = json_data.get("NODE_LIST", [])
    final_output = json_data # 원본 구조 유지
elif isinstance(json_data, list):
    node_list = json_data
    final_output = {"NODE_LIST": json_data} # NODE_LIST 키로 감싸기
else:
    node_list = []
    final_output = {"NODE_LIST": []}

matched_count = 0

for paper in node_list:
    node_id = paper.get("NODE_ID")
    
    if node_id in paper_author_map:
        author_ids = paper_author_map[node_id]
        
        # 저자 ID 리스트를 쉼표로 연결된 문자열로 변환
        paper["AUTR_ID"] = ",".join(map(str, author_ids))
        matched_count += 1
    else:
        paper["AUTR_ID"] = ""

# 4. 결과 저장 (가독성 향상: indent=4 적용)
output_filename = 'SSU_Datathon2025_공학분야_62199_with_AUTR_ID.json'

with open(output_filename, 'w', encoding='utf-8') as f:
    # indent=4: 들여쓰기를 4칸씩 적용하여 줄바꿈 처리 (가독성 확보)
    # ensure_ascii=False: 한글이 깨지지 않고 그대로 저장됨
    json.dump(final_output, f, ensure_ascii=False, indent=4)

print("작업 완료!")
print(f"총 {matched_count}개의 논문에 아이디가 추가되었습니다.")
print(f"저장된 파일: {output_filename}")
print("형식: 들여쓰기가 적용된 보기 편한 JSON 포맷으로 저장되었습니다.")

작업 완료!
총 35524개의 논문에 아이디가 추가되었습니다.
저장된 파일: SSU_Datathon2025_공학분야_62199_with_AUTR_ID.json
형식: 들여쓰기가 적용된 보기 편한 JSON 포맷으로 저장되었습니다.


In [4]:
# AUTR_ID 에서 공백인 논문 추출

import json
import os

# 1. 파일 경로 설정
# 방금 전 단계에서 생성한 파일명
input_filename = 'SSU_Datathon2025_공학분야_62199_with_AUTR_ID.json'
output_filename = 'no_AUTR_ID_papers.json'

# 파일 확인
if not os.path.exists(input_filename):
    print(f"오류: '{input_filename}' 파일이 없습니다. 이전 단계 코드를 먼저 실행해서 파일을 만들어주세요.")
    exit()

# 2. JSON 파일 읽기
print("파일을 읽는 중...")
with open(input_filename, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# 3. 필터링: AUTR_ID가 "" (공백)인 것만 리스트 컴프리헨션으로 추출
# 코드가 훨씬 짧고 빠릅니다.
node_list = json_data.get("NODE_LIST", [])
unmatched_list = [paper for paper in node_list if paper.get("AUTR_ID") == ""]

# 4. 결과 저장 (논문 1개당 1줄씩 포맷팅)
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write('{ "UNMATCHED_LIST": [\n')
    
    total_items = len(unmatched_list)
    for i, paper in enumerate(unmatched_list):
        line = json.dumps(paper, ensure_ascii=False)
        
        # 마지막 항목인지 체크해서 콤마(,) 결정
        if i < total_items - 1:
            f.write('  ' + line + ',\n')
        else:
            f.write('  ' + line + '\n')
            
    f.write(']}')

# 5. 결과 출력
print("-" * 30)
print(f"작업 완료!")
print(f"전체 {len(node_list)}개 중 AUTR_ID가 없는 논문 {len(unmatched_list)}개를 추출했습니다.")
print(f"저장된 파일: {output_filename}")
print("-" * 30)

파일을 읽는 중...
------------------------------
작업 완료!
전체 35790개 중 AUTR_ID가 없는 논문 266개를 추출했습니다.
저장된 파일: no_AUTR_ID_papers.json
------------------------------


In [5]:
# 매칭 안된 논문 중 저자가 '편집부'인 경우

import json
import os
from collections import Counter

# 1. 파일 경로 (방금 만든 매칭 실패 파일)
filename = 'no_AUTR_ID_papers.json'

if not os.path.exists(filename):
    print(f"오류: '{filename}' 파일이 없습니다. 경로를 확인해주세요.")
else:
    with open(filename, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    # 2. 리스트 가져오기
    # 파일 생성 방식에 따라 키값이 'UNMATCHED_LIST' 혹은 'NODE_LIST'일 수 있어 둘 다 체크합니다.
    paper_list = json_data.get("UNMATCHED_LIST", json_data.get("NODE_LIST", []))

    # 3. 개수 세기
    total_count = len(paper_list)
    editorial_count = 0
    
    # '편집부' 외에 다른 유형도 확인하기 위해 이름들을 모아봅니다.
    author_names = []

    for paper in paper_list:
        # 저자명이 없는 경우 빈 문자열로 처리
        author_name = paper.get("AUTR_NM", "")
        if author_name is None: author_name = ""
        
        author_names.append(author_name)

        # "편집부"라는 글자가 이름에 포함되어 있으면 카운트
        if "편집부" in author_name:
            editorial_count += 1

    # 4. 결과 출력
    print("=" * 40)
    print(f"📂 파일명: {filename}")
    print(f"📊 전체 미매칭 논문 수: {total_count}개")
    print(f"✅ '편집부' 포함 저자 수: {editorial_count}개")
    print(f"   (비율: {editorial_count / total_count * 100:.2f}%)")
    print("=" * 40)

    # [추가 팁] 편집부 말고 가장 많이 등장하는 이름 TOP 5 확인
    # 이걸 보면 '편집부' 말고 '학회', '사무국' 등이 얼마나 있는지 알 수 있습니다.
    print("[참고] 미매칭 파일 내 최다 등장 이름 TOP 5:")
    top_authors = Counter(author_names).most_common(5)
    for name, count in top_authors:
        print(f" - {name if name else '(이름 없음)'}: {count}개")

📂 파일명: no_AUTR_ID_papers.json
📊 전체 미매칭 논문 수: 266개
✅ '편집부' 포함 저자 수: 265개
   (비율: 99.62%)
[참고] 미매칭 파일 내 최다 등장 이름 TOP 5:
 - 편집부: 265개
 - 곽호찬, 김흥기, 양신추, 박호철, 이환승, 백승구, 김성, 김현승, 장태우, 이명원, 오석문: 1개


In [8]:
# 아이디를 기준으로 논문 수 내림차순으로 정리

import pandas as pd
import json
import os
from collections import Counter

# 1. 파일 경로 설정
# 아까 한글 이름 우선으로 정리해서 저장했던 그 파일명을 넣으세요.
clean_csv_filename = 'rep_rem_authors_in_korean.csv' 
json_filename = 'SSU_Datathon2025_공학분야_62199_with_AUTR_ID.json'

# 파일이 있는지 확인
if not os.path.exists(clean_csv_filename):
    print(f"오류: '{clean_csv_filename}' 파일이 없습니다. 파일명을 확인해주세요.")
    exit()

# 2. 정리된 CSV 파일 불러오기 (이미 깔끔한 상태)
print("1. 정리된 저자 목록(CSV)을 불러옵니다...")
df_clean = pd.read_csv(clean_csv_filename)

# 바로 매핑 사전으로 변환 (복잡한 정렬/필터링 과정 생략 가능!)
# { '12345': '홍길동', ... }
id_name_map = dict(zip(df_clean['Author ID'].astype(str), df_clean['Author Name']))

# 3. JSON 파일에서 논문 수 카운트
print("2. JSON 파일에서 아이디별 논문 수를 셉니다...")
with open(json_filename, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

all_author_ids = []
node_list = json_data.get("NODE_LIST", [])

for paper in node_list:
    ids_str = paper.get("AUTR_ID", "")
    if ids_str:
        # 아이디가 여러 개면 쉼표(,)로 쪼개서 리스트에 넣음
        all_author_ids.extend(ids_str.split(","))

# 개수 세기
id_counts = Counter(all_author_ids)

# 4. 결과 합치기
print("3. 최종 결과 파일을 만듭니다...")
result_list = []

for author_id, count in id_counts.items():
    if not author_id: continue # 빈 아이디는 패스

    # 아까 불러온 파일에서 이름 찾기 (없으면 Unknown)
    name = id_name_map.get(author_id, "Unknown")
    
    result_list.append({
        'Author Name': name,
        'Author ID': author_id,
        'Paper Count': count
    })

# 데이터프레임 변환
df_result = pd.DataFrame(result_list)

# 5. 내림차순 정렬 및 저장
df_result = df_result.sort_values(by='Paper Count', ascending=False)

output_filename = 'author_paper_counts.csv'
df_result.to_csv(output_filename, index=False, encoding='utf-8-sig')

print("-" * 30)
print("작업 완료!")
print(f"총 {len(df_result)}명의 저자 통계가 저장되었습니다.")
print(f"상위 3명 예시:")
print(df_result.head(3))
print(f"저장된 파일: {output_filename}")

1. 정리된 저자 목록(CSV)을 불러옵니다...
2. JSON 파일에서 아이디별 논문 수를 셉니다...
3. 최종 결과 파일을 만듭니다...
------------------------------
작업 완료!
총 58738명의 저자 통계가 저장되었습니다.
상위 3명 예시:
      Author Name  Author ID  Paper Count
5761          허장욱  647329504           67
14110         이상진     562214           60
4             이수기  751385579           52
저장된 파일: author_paper_counts.csv
